# NLP-03 · LLM Training Pipeline
### Section 05 — Modern NLP and LLMs · *ML/AI Senior Mastery Curriculum*

> How does a language model go from a random initialisation to writing code, passing
> exams, and following nuanced instructions? This notebook traces the **four-stage
> pipeline** used to build every modern LLM: **pre-training** (learn language from
> text) → **continual pre-training** (adapt to a domain) → **supervised fine-tuning
> / SFT** (learn to follow instructions) → **alignment** (RLHF or DPO — optimize
> measured preferences and constraints). This notebook maps the lifecycle after the
> student has trained the real decoder in `projects/tiny_language_model`. BPE is
> implemented fully; SFT, DPO, and LoRA use deliberately small arithmetic
> demonstrations. Those demonstrations are not completed model-training runs.


> **Canonical learner route · module NLP-03 of 62**
>
> Required prerequisites: **FND-04, DL-08, NLP-01, DL-06, DL-07** · Previous: **NLP-02** ·
> Next after mastery: **EVAL-02** · Expected first-pass workload:
> **5–8 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` is authoritative when section-local file order and
> prerequisite order differ.


## 1 · Learning Objectives

**What you will master**
- **BPE tokenisation** from scratch: why subword tokenisation, the merge algorithm,
  vocabulary size tradeoffs.
- **Pre-training objective**: causal language modelling (next-token prediction
  cross-entropy), teacher forcing, compute budgeting via Chinchilla scaling laws
  (Lesson DL-08).
- **Data curation** for pre-training: deduplication, quality filtering, domain
  mixing, data poisoning risks.
- **Supervised fine-tuning (SFT)**: instruction-response templates (Alpaca, ChatML,
  Llama-2), loss masking on prompt tokens.
- **RLHF** (PPO): reward model training, the PPO loop, KL penalty against the
  reference model.
- **DPO** (Direct Preference Optimisation): no reward model needed, just preference
  pairs — the Bradley-Terry model and the DPO loss.
- **LoRA** (Low-Rank Adaptation): freeze base weights and add low-rank adapters to
  selected projections; measure trainable state and retention rather than assuming it.

**Why it matters**
- Understanding the training pipeline is prerequisite to: fine-tuning LLMs for
  your domain (Lesson PROD-01 MLOps), evaluating them (EVAL-02, EVAL-04, and EVAL-05), and
  diagnosing production failures (Lesson NLP-05 Hallucinations).

**Typical interview questions**
- "Explain BPE tokenisation."
- "What is the difference between SFT and RLHF?"
- "Why is DPO simpler than RLHF? What does it sacrifice?"
- "How does LoRA work, and why must retention still be evaluated?"


<details>
<summary><strong>Mathematics notation support — open when a formula feels dense</strong></summary>

- $x_i$: item or component number $i$; a subscript is a label, not multiplication.
- $n$: number of examples; $d$: number of features or dimensions.
- $\mathbf x$: vector; $X$: matrix; $X^\top$: transpose (rows and columns swap).
- $\hat y$: an estimate or prediction; a hat marks an estimated quantity.
- $\sum$: add repeated terms; $\prod$: multiply repeated terms.
- $\lVert\mathbf x\rVert$: vector length; $|x|$: distance of one number from zero.
- $\frac{\partial f}{\partial x}$: slope of $f$ as $x$ changes while other inputs
  stay fixed; $\nabla f$: vector containing all parameter slopes.
- $P(A\mid B)$: probability of $A$ after restricting attention to cases where
  $B$ is true.
- $\log$ reverses an exponential and turns products into sums.

Read a formula one operator at a time, write object shapes beside vectors and
matrices, and substitute a tiny numeric example. Review PRE-01 through PRE-04 for
worked explanations of these symbols.
</details>


## Student Lesson Companion · NLP-03 · LLM Training Pipeline

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |

### Code walkthrough and expected-result contract

For the first implementation, annotate: inputs → shapes/units → initialization →
central computation → intermediate output → final output → verification. Before
execution, record the expected value, range, or shape and what it would mean.

Distinguish these outcomes:

| Outcome | Interpretation | Next action |
|---|---|---|
| Exception, non-finite value, impossible shape | Code or data-contract failure | Inspect the first violated boundary |
| Output has valid type/shape but weak metric | Experiment ran; method may be poor | Diagnose data, assumptions, and baseline comparison |
| Strong metric on training data only | Insufficient evidence | Evaluate with the declared validation design |
| Expected output on untouched data | Supports one scoped claim | Record limitations; do not generalize beyond evidence |

### Debugging table

| Symptom | Likely cause | Inspect | Scoped fix |
|---|---|---|---|
| Shape/type error | Interface mismatch | Shapes, dtypes, feature names | Repair the boundary, not downstream symptoms |
| NaN/Inf or divergence | Invalid input or unstable update | Raw values, loss, gradients, learning rate | Clean/validate input or change one optimization control |
| Implausibly strong result | Leakage or invalid split | Fit boundaries, timestamps, duplicate entities | Rebuild the evaluation path before tuning |
| Different repeated result | Uncontrolled state | Seeds, data order, train/eval mode, versions | Record and control randomness intentionally |
| Plausible output but poor result | Wrong assumptions or representation | Baseline, slices, residuals/errors | Prefer a justified alternative; do not debug valid code as broken |


### Required entry evidence: train a language model before adapting one

Do not begin this pipeline as an API-only learner. First complete
`projects/tiny_language_model/MASTERY_CHECKPOINT.md`. You should be able to point to
a real checkpoint and show tokenization, shifted targets, causal masking, forward
loss, backward gradients, AdamW updates, validation loss, and generation.

| Evidence label | What it supports |
|---|---|
| **Executed training** | Parameters were updated and measured on held-out data |
| **Arithmetic demonstration** | A formula behaves as described on controlled numbers |
| **Schematic** | A relationship is visualized but not estimated here |
| **Tool pattern** | Integration code whose result needs a declared model and data |

The random-logit SFT and DPO cells below are arithmetic demonstrations. They cannot
establish convergence, alignment quality, safety, or production readiness.


## 2 · Historical Motivation

**GPT (2018): pre-training works.** Radford et al. showed that a Transformer
trained on next-token prediction on BooksCorpus could be fine-tuned to beat
task-specific models on diverse NLP benchmarks. The key insight: language
modelling is a universal pre-training objective.

**GPT-2 and scaling (2019).** Scaled to 1.5B parameters, the model showed
surprising zero-shot capability. The paper introduced the idea that LLMs may be
"multitask learners" without task-specific fine-tuning.

**GPT-3 and in-context learning (2020).** 175B parameters trained on 300B tokens.
Introduced few-shot learning purely via the prompt — no gradient update. But the
raw model was difficult to use: it completed text rather than following instructions.

**InstructGPT and RLHF (2022).** Ouyang et al. (OpenAI) showed that fine-tuning
GPT-3 with SFT + RLHF on human-labelled data produced a much more helpful and
safer model despite being 100× smaller. This established the **SFT → RLHF**
recipe.

**Stanford Alpaca (2023)** democratised instruction tuning: fine-tune LLaMA-7B on
52K GPT-4-generated instruction-response pairs for ~$600. Spawned the instruction-
tuning cottage industry.

**DPO (Rafailov et al., 2023)** offers a simpler offline preference-optimization
route for a particular regularized objective: derive that the optimal RLHF
policy implicitly defines a reward, so you can optimise it directly from preference
pairs without training a separate reward model or running PPO. This removes parts
of the RLHF pipeline but introduces its own data, reference-policy, and tuning risks.

**LoRA (Hu et al., 2021)** made fine-tuning affordable: instead of updating all
$d \times k$ parameters of a weight matrix, add two low-rank matrices $A$ ($d
\times r$) and $B$ ($r \times k$) and train only those. $r=8$ or $r=16$ is typical;
for a 7B model this reduces trainable parameters from 7B to ~4M.


## 3 · Intuition & Visual Understanding

**The four-stage pipeline.**

```mermaid
flowchart LR
    PT["1. Pre-training\nTrillions of tokens\nNext-token prediction\n(Chinchilla budget)"]
    CPT["2. Continual Pre-training\nDomain corpus\n(optional, cheap)"]
    SFT["3. SFT\nInstruction-response pairs\n(10K-1M examples)\nLoss masked on prompt"]
    ALIGN["4. Alignment\nRLHF (PPO+reward model)\nor DPO (preference pairs)"]
    PT --> CPT --> SFT --> ALIGN
```

**BPE intuition.** Start with character-level tokens. Find the most frequent pair
of adjacent tokens and merge them into a new token. Repeat until the vocabulary
reaches the target size. Result: common words → single tokens; rare/OOV words →
subword pieces; characters → last-resort fallback. "tokenization" might become
["token", "ization"] or ["tok", "en", "ization"] depending on training corpus.

**LoRA intuition.** A pre-trained weight matrix $W_0 \in \mathbb{R}^{d \times k}$
stores everything the model learned about language. Fine-tuning wants to add a
small correction $\Delta W$. Using the row-vector convention in this notebook,
LoRA assumes $\Delta W = AB$ where $A \in \mathbb{R}^{d \times r}$ and
$B \in \mathbb{R}^{r \times k}$ with $r \ll \min(d,k)$. Instead
of storing $dk$ floats for $\Delta W$, we store only $r(d+k)$ — with $r=8$,
$d=k=4096$, that's 65K vs 16M. The hypothesis: the *task adaptation* of a large
model may be approximated in a low-dimensional subspace. Rank is a measured
capacity choice, not a guarantee.

**DPO intuition.** RLHF trains a reward model $r_\phi$ and then optimises the
policy with PPO. DPO notices that the optimal policy under the RLHF objective is:
$\pi^*(y|x) \propto \pi_{\text{ref}}(y|x)\,\exp(r(x,y)/\beta)$. Plugging this
back in, we can express the reward implicitly via the policy itself, eliminating
the need for $r_\phi$ entirely — you just need preference pairs $(y_w, y_l)$ per
prompt $x$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter, defaultdict

rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

CORPUS_SAMPLE = [
    "the cat sat on the mat",
    "the cat ate the rat",
    "tokenization splits words into subwords",
    "low rank adaptation is efficient",
    "language models are trained on tokens",
    "the dog ran after the cat",
    "supervised fine tuning follows pre training",
    "reinforcement learning from human feedback",
]
print(f"Sample corpus: {len(CORPUS_SAMPLE)} sentences")


## 4 · Mathematical Foundations

### 4.1 BPE tokenisation

**Algorithm.** Given a corpus:
1. Initialise vocabulary as character-level tokens + `<EOS>`.
2. Represent each word as a sequence of characters separated by spaces (plus a
   word-boundary marker `</w>` at the end).
3. Count all adjacent symbol pairs across the corpus.
4. Merge the most frequent pair → new symbol. Add to vocabulary.
5. Repeat steps 3–4 until target vocab size is reached or no pairs remain.

**Encoding a new word** (after training): greedily apply the learned merge rules
in order. Any character not in the vocabulary maps to `<UNK>` (or is handled by
byte-level BPE as in GPT-4).

### 4.2 Pre-training loss (causal LM)

For a sequence $x_1, x_2, \dots, x_T$ and model parameters $\theta$:
$$J_{\text{CLM}} = -\frac{1}{T-1} \sum_{t=1}^{T-1}
  \log p_\theta(x_{t+1} \mid x_1, \dots, x_t)$$
This is exactly the cross-entropy from Lesson FND-02 applied position-by-position,
with a **causal mask** preventing the model from seeing future tokens (Lesson DL-08).

### 4.3 SFT loss (instruction masking)

For an instruction-response pair (prompt $P$, response $R$), we concatenate to
form $[P; R]$ and apply CLM loss only on the response tokens:
$$J_{\text{SFT}} = -\frac{1}{|R|} \sum_{t \in R} \log p_\theta(x_t \mid x_{<t})$$
The prompt tokens contribute to the context but NOT to the loss. This prevents the
model from "forgetting" how to follow format while learning to generate responses.

### 4.4 RLHF objective

Train a reward model $r_\phi$ to predict human preference:
$$J_{\text{RM}} = -\mathbb{E}_{(x,y_w,y_l)}\!\left[
  \log\sigma(r_\phi(x,y_w) - r_\phi(x,y_l))\right]$$
(Bradley-Terry preference model). Then optimise the policy $\pi_\theta$:
$$J_{\text{PPO}} = \mathbb{E}[r_\phi(x,y)] - \beta\,\mathrm{KL}\!\left[\pi_\theta(y|x) \| \pi_{\text{ref}}(y|x)\right]$$
The KL term prevents the policy from drifting too far from the supervised baseline.

### 4.5 DPO loss

Directly optimise preference pairs without a reward model:
$$J_{\text{DPO}} = -\mathbb{E}_{(x,y_w,y_l)}\!\left[\log\sigma\!\left(
  \beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)}
  - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$
Intuition: increase the log-probability of $y_w$ (winner) relative to the
reference model, while decreasing it for $y_l$ (loser).

### 4.6 LoRA

Replace a forward pass $h = xW_0$ with:
$$h = xW_0 + x\underbrace{AB}_{\Delta W},\quad A \in \mathbb{R}^{d \times r},\ B \in \mathbb{R}^{r \times k}$$
Initialise $B=0$, $A \sim \mathcal{N}(0, \sigma^2)$ so $\Delta W = 0$ at start
(no change to pre-trained behaviour). Scale by $\alpha/r$ (hyperparameter $\alpha$).
Only $A$ and $B$ are trained; $W_0$ is frozen.


## 5 · Manual Implementation from Scratch

### 5a BPE tokenisation from scratch


In [ ]:
# 5a. BPE implementation from scratch in pure Python.
def get_pairs(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    bigram = re.escape(" ".join(pair))
    pattern = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
    new_vocab = {}
    for word, freq in vocab.items():
        new_word = pattern.sub("".join(pair), word)
        new_vocab[new_word] = freq
    return new_vocab

def learn_bpe(corpus_sentences, num_merges=30):
    # Build initial vocab: characters + </w> as word boundary.
    vocab = Counter()
    for sent in corpus_sentences:
        for word in sent.split():
            word_str = " ".join(list(word)) + " </w>"
            vocab[word_str] += 1
    merges = []
    for i in range(num_merges):
        pairs = get_pairs(vocab)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)
        vocab = merge_vocab(best, vocab)
    return vocab, merges

# Train BPE on our sample corpus.
bpe_vocab, bpe_merges = learn_bpe(CORPUS_SAMPLE, num_merges=25)
print(f"BPE merges learned: {len(bpe_merges)}")
print(f"Final BPE tokens (sample):")
for token, freq in sorted(bpe_vocab.items(), key=lambda x: -x[1])[:8]:
    print(f"  '{token}'  freq={freq}")


In [ ]:
# 5a.2 Encode a new sentence using learned BPE merge rules.
def apply_bpe(word, merges):
    chars = list(word) + ["</w>"]
    tokens = chars[:]
    for pair in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(tokens[i] + tokens[i + 1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

def tokenise_bpe(sentence, merges):
    result = []
    for word in sentence.lower().split():
        result.extend(apply_bpe(word, merges))
    return result

test_sentences = [
    "the cat sat",
    "tokenization is important",
    "low rank finetuning",
]
for s in test_sentences:
    tokens = tokenise_bpe(s, bpe_merges)
    print(f"  '{s}' -> {tokens}")


### 5b Pre-training loss arithmetic (causal LM cross-entropy)

This controlled calculation verifies the loss formula. The executed pre-training
run and held-out learning curve are in `projects/tiny_language_model`.


In [ ]:
# 5b. Implement the causal LM loss from scratch.
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def causal_lm_loss(logits, targets):
    # logits: (T, V), targets: (T,) integer token ids.
    # Loss on all T tokens (pre-training).
    T, V = logits.shape
    probs = softmax(logits)
    log_probs = np.log(probs[np.arange(T), targets] + 1e-12)
    return -log_probs.mean()

# Simulate: tiny 8-token sequence, 10-token vocabulary, random logits.
V_sim, T_sim = 10, 8
logits_random = rng.normal(0, 1, (T_sim, V_sim))
targets_sim   = rng.integers(0, V_sim, T_sim)
loss_random   = causal_lm_loss(logits_random, targets_sim)
# Expected loss at random init ~ log(V).
print(f"Random init loss: {loss_random:.3f}  (expected ≈ log({V_sim}) = {np.log(V_sim):.3f})")
# Simulate near-perfect logits.
logits_good = np.eye(V_sim)[targets_sim] * 10
loss_good   = causal_lm_loss(logits_good, targets_sim)
print(f"Near-perfect logits loss: {loss_good:.3f}")


### 5c SFT loss arithmetic — mask direct prompt-token losses


In [ ]:
# 5c. SFT loss: loss only on response tokens, prompt tokens masked out.
def sft_loss(logits, targets, prompt_len):
    # prompt_len: number of tokens to EXCLUDE from loss.
    T, V = logits.shape
    assert prompt_len < T
    probs = softmax(logits)
    # Only compute loss on response tokens (from prompt_len onwards).
    resp_logits  = logits[prompt_len:]
    resp_targets = targets[prompt_len:]
    resp_probs   = softmax(resp_logits)
    log_probs    = np.log(resp_probs[np.arange(len(resp_targets)), resp_targets] + 1e-12)
    return -log_probs.mean()

# Simulate prompt (4 tokens) + response (4 tokens).
logits_sft  = rng.normal(0, 1, (8, V_sim))
targets_sft = rng.integers(0, V_sim, 8)
loss_clm_all = causal_lm_loss(logits_sft, targets_sft)
loss_sft_only = sft_loss(logits_sft, targets_sft, prompt_len=4)
print(f"CLM loss (all 8 tokens):       {loss_clm_all:.3f}")
print(f"SFT loss (response 4 tokens):  {loss_sft_only:.3f}")
print("The SFT loss only penalises wrong predictions on the RESPONSE side.")
print("Prompt positions have no direct loss term, but their representations condition")
print("response predictions, so gradients can still flow through the prompt context.")


### 5d DPO objective arithmetic


In [ ]:
# 5d. DPO loss from scratch (batched).
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def log_prob_sequence(logits, targets):
    # Sum of log probs over a response sequence.
    probs = softmax(logits)
    return np.sum(np.log(probs[np.arange(len(targets)), targets] + 1e-12))

def dpo_loss(logits_w, logits_l, logits_w_ref, logits_l_ref,
             targets_w, targets_l, beta=0.1):
    # w = winner (preferred), l = loser (rejected).
    lp_w     = log_prob_sequence(logits_w,     targets_w)
    lp_l     = log_prob_sequence(logits_l,     targets_l)
    lp_w_ref = log_prob_sequence(logits_w_ref, targets_w)
    lp_l_ref = log_prob_sequence(logits_l_ref, targets_l)
    # Log-ratio of policy vs reference.
    ratio_w = lp_w - lp_w_ref
    ratio_l = lp_l - lp_l_ref
    return -np.log(sigmoid(beta * (ratio_w - ratio_l)))

# Simulate: 4 response tokens, 10 vocab.
T_dpo = 4
# "Good" policy: high logits for winner tokens, low for loser.
targets_w_dpo = rng.integers(0, V_sim, T_dpo)
targets_l_dpo = rng.integers(0, V_sim, T_dpo)
logits_w_policy  = np.eye(V_sim)[targets_w_dpo] * 5        # policy prefers winner
logits_l_policy  = rng.normal(0, 1, (T_dpo, V_sim))        # policy uncertain on loser
logits_w_ref     = rng.normal(0, 1, (T_dpo, V_sim))        # reference uncertain
logits_l_ref     = rng.normal(0, 1, (T_dpo, V_sim))
loss_dpo = dpo_loss(logits_w_policy, logits_l_policy,
                    logits_w_ref,    logits_l_ref,
                    targets_w_dpo,   targets_l_dpo)
print(f"DPO loss (policy prefers winner): {loss_dpo:.3f}  (lower is better)")
# Bad policy: uncertain on winner targets and confident on loser targets.
logits_w_bad = rng.normal(0, 1, (T_dpo, V_sim))
logits_l_bad = np.eye(V_sim)[targets_l_dpo] * 5
loss_dpo_bad = dpo_loss(logits_w_bad, logits_l_bad,
                        logits_w_ref,    logits_l_ref,
                        targets_w_dpo,   targets_l_dpo)
print(f"DPO loss (policy prefers loser):  {loss_dpo_bad:.3f}  (should be high)")


### 5e LoRA forward-pass arithmetic


In [ ]:
# 5e. LoRA: low-rank adaptation of a weight matrix.
class LoRALinear:
    def __init__(self, d_in, d_out, rank=4, alpha=8):
        self.W0 = rng.normal(0, 0.02, (d_in, d_out))   # pretrained, frozen
        # LoRA adapters: A initialized with small random; B initialized to 0.
        self.A  = rng.normal(0, 0.01, (d_in, rank))
        self.B  = np.zeros((rank, d_out))
        self.scale = alpha / rank                       # LoRA scaling factor

    def forward(self, x):
        base   = x @ self.W0                           # pretrained path (no grad)
        lora   = (x @ self.A) @ self.B * self.scale   # adapter path (trained)
        return base + lora                             # additive merge

    def param_count(self):
        total   = self.W0.size
        adapter = self.A.size + self.B.size
        return total, adapter, adapter / total

layer = LoRALinear(d_in=512, d_out=512, rank=8)
total, adapter, frac = layer.param_count()
print(f"Weight matrix params:   {total:,}")
print(f"LoRA adapter params:    {adapter:,}")
print(f"Fraction trained:       {frac*100:.2f}%")
print()
x_test = rng.normal(0, 1, (4, 512))          # batch of 4 tokens
out = layer.forward(x_test)
print(f"LoRA output shape: {out.shape}")
print("At init B=0, so out == x @ W0 (no change to base model behaviour)")
print(f"Max |delta| vs base: {np.abs(out - x_test @ layer.W0).max():.6f}")


In [ ]:
# LoRA parameter savings across common model sizes.
configs = [
    ("7B (d=4096, r=8)",  4096, 4096, 8),
    ("13B (d=5120, r=8)", 5120, 5120, 8),
    ("70B (d=8192, r=16)",8192, 8192, 16),
]
print(f"{'Config':<25} {'W0 params':>12} {'LoRA params':>12} {'Ratio':>8}")
for name, d_in, d_out, r in configs:
    w0 = d_in * d_out
    lo = d_in * r + r * d_out
    print(f"{name:<25} {w0:>12,} {lo:>12,} {lo/w0:>7.3%}")
print("\nWith ~32 attention projection layers per 7B model,")
print("LoRA r=8 trains ~4M params out of 7B = 0.06% of all weights.")


## 6 · Visualization


In [ ]:
# Figure 1 — BPE merge frequency: how many pairs are merged at each step.
bpe_freq_all = []
vocab_temp = Counter()
for sent in CORPUS_SAMPLE:
    for word in sent.split():
        vocab_temp[" ".join(list(word)) + " </w>"] += 1
for i in range(20):
    pairs = get_pairs(vocab_temp)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    bpe_freq_all.append(pairs[best])
    vocab_temp = merge_vocab(best, vocab_temp)

fig, ax = plt.subplots()
ax.bar(range(len(bpe_freq_all)), bpe_freq_all, color="steelblue")
ax.set_xlabel("merge step"); ax.set_ylabel("frequency of merged pair")
ax.set_title("Figure 1 — BPE: merge frequency decreases as common pairs are exhausted")
plt.show()


**Figure 1.** The frequency of the most-common pair merged at each BPE step.
Early merges are high-frequency pairs ("t h" → "th", "a t" → "at") that appear
across many words. Later merges combine rarer subwords. This diminishing-frequency
curve reflects the Zipfian distribution of natural language: a few patterns are
very common, and the long tail is rare. In production (GPT-4 uses cl100k with
~100K merges), the curve is much smoother because the training corpus is orders
of magnitude larger.


In [ ]:
# Figure 2 — Loss landscape: pre-training → SFT → alignment (schematic).
stages = ["Random\ninit", "Mid\npre-train", "Pre-train\ndone", "Post\nSFT", "Post\nRLHF/DPO"]
clm_loss   = [np.log(50257), 3.5, 2.0, None, None]          # GPT-2 vocab size
sft_loss_v = [None, None, 2.0, 1.2, None]
align_r    = [None, None, None, 5.5, 8.2]                   # reward (higher=better)

x = np.arange(len(stages))
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

valid_clm = [(i, v) for i, v in enumerate(clm_loss) if v is not None]
valid_sft = [(i, v) for i, v in enumerate(sft_loss_v) if v is not None]
valid_al  = [(i, v) for i, v in enumerate(align_r) if v is not None]

ax1.plot([i for i, v in valid_clm], [v for i, v in valid_clm], "o-b", label="CLM loss")
ax1.plot([i for i, v in valid_sft], [v for i, v in valid_sft], "s--g", label="SFT loss")
ax2.plot([i for i, v in valid_al],  [v for i, v in valid_al],  "^-r", label="Reward")
ax1.set_xticks(x); ax1.set_xticklabels(stages, fontsize=8)
ax1.set_ylabel("loss (lower is better)", color="blue")
ax2.set_ylabel("reward (higher is better)", color="red")
ax1.set_title("Figure 2 — LLM training stages: loss and reward across the pipeline")
lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, loc="center right")
plt.tight_layout()
plt.show()


**Figure 2.** A schematic of the LLM training pipeline. **CLM loss** (blue)
starts near $\log(V) \approx 10.8$ for GPT-2's 50K vocabulary (random init) and
falls to ~2.0 after pre-training — the model has learned English grammar and world
knowledge. **SFT** (green) fine-tunes on instruction-response pairs; the loss
drops further because the task is narrower than all of language. **Alignment**
(RLHF/DPO, red) doesn't necessarily reduce loss — it optimises a **reward** score
(human preference), which can sometimes trade off perplexity for helpfulness.
This is why alignment is a separate, complementary stage, not a continuation of
language modelling.


In [ ]:
# Figure 3 — LoRA parameter efficiency: trainable vs frozen params by rank.
ranks = [1, 2, 4, 8, 16, 32, 64]
d = 4096
total_w = d * d
trainable = [d * r + r * d for r in ranks]
fractions = [t / total_w * 100 for t in trainable]
fig, ax = plt.subplots()
ax.semilogx(ranks, fractions, "o-", base=2)
ax.set_xlabel("LoRA rank r (log2 scale)"); ax.set_ylabel("% trainable params (of one W matrix)")
ax.set_title(f"Figure 3 — LoRA rank vs parameter budget (d={d}x{d} matrix)")
for r, f in zip(ranks, fractions):
    ax.annotate(f"{f:.2f}%", (r, f), textcoords="offset points", xytext=(5, 0), fontsize=8)
plt.show()


**Figure 3.** LoRA rank vs the percentage of trainable parameters for a single
$4096 \times 4096$ weight matrix. At rank $r=8$ (the standard default), LoRA uses
only 0.39% of the parameters of that matrix — yet retains >95% of the fine-tuning
quality on most tasks. Increasing rank trades efficiency for expressiveness: $r=64$
approaches the quality of full fine-tuning. The typical production choice is
$r \in \{8, 16\}$ with $\alpha = 2r$ (so the effective scale is ~2). Applied to
all 32+ attention projections in a 7B model, this remains well under 1% of total
parameters.


In [ ]:
# Figure 4 — DPO loss as a function of the log-ratio margin.
margins = np.linspace(-3, 3, 200)               # beta*(ratio_w - ratio_l)
dpo_loss_curve = -np.log(sigmoid(margins))
grad_magnitude  = sigmoid(-margins)              # gradient wrt margin
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.plot(margins, dpo_loss_curve, "b-", label="DPO loss")
ax2.plot(margins, grad_magnitude, "r--", label="gradient magnitude")
ax1.set_xlabel("beta * (log-ratio winner - log-ratio loser)")
ax1.set_ylabel("DPO loss", color="blue")
ax2.set_ylabel("gradient magnitude", color="red")
ax1.set_title("Figure 4 — DPO loss and gradient: easy vs hard preferences")
lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labs1+labs2)
plt.show()


**Figure 4.** The DPO loss (blue) as a function of the margin
$\beta(\text{ratio}_w - \text{ratio}_l)$. When the policy strongly prefers the
winner (positive margin), the loss is near zero and the gradient (red, dashed) is
tiny — these "easy" preferences are already satisfied and contribute little to
training. When the policy prefers the loser (negative margin), the loss is high
and the gradient is large — the objective applies a stronger correction. This
curve explains the local arithmetic; it does not establish that DPO will converge
faster or more reliably than PPO on a real model. If the margin is very negative
prefers loser), the gradient *still* saturates — DPO doesn't give special
treatment to very wrong answers, which can be a failure mode (§7).


## 7 · Failure Modes

| Failure | Symptom | Root cause | Mitigation |
|---|---|---|---|
| **Catastrophic forgetting** | Model loses pre-trained capability after adaptation | Updates change useful behavior | Retention set; small LR; replay; compare LoRA and full tuning |
| **Reward hacking (RLHF)** | Policy learns to fool reward model | Reward model not robust; KL penalty too weak | Stronger KL; better RM; constitutional AI |
| **DPO collapse** | Policy ignores loser, doesn't learn winner | Margin saturates both sides | IPO/cDPO; add KL term; higher beta |
| **Data contamination** | Inflated eval scores | Test prompts in SFT data | Dedup train vs eval; time-split evals |
| **Mode collapse (SFT)** | Model gives identical outputs | Over-fitting small SFT set; LR too high | Lower LR; early stopping; diverse data |
| **BPE vocabulary mismatch** | OOV subwords at inference | Tokeniser trained on different domain | Fine-tune tokeniser or use byte-level BPE |
| **Length bias (RM)** | Reward model prefers longer answers | Length correlated with perceived quality in training | Length-penalised reward; length-normalised reward |
| **SFT format sensitivity** | Model breaks if prompt format changes | Format memorised, not generalised | Data augmentation with format variants |


In [ ]:
# Demonstrate: catastrophic forgetting via full update vs LoRA.
# We measure how much the output changes after full vs LoRA update.
d_demo = 64
W_pretrained = rng.normal(0, 0.02, (d_demo, d_demo))
x_demo = rng.normal(0, 1, (8, d_demo))
out_pretrained = x_demo @ W_pretrained

# Full fine-tune: update all weights (large step simulating over-fitting).
W_full = W_pretrained + rng.normal(0, 0.5, W_pretrained.shape)   # large perturbation
out_full = x_demo @ W_full
delta_full = np.abs(out_full - out_pretrained).mean()

# LoRA: only adapters change (B still 0 at init, small A step).
A = rng.normal(0, 0.01, (d_demo, 4))
B = rng.normal(0, 0.5, (4, d_demo))    # even large B perturbation
out_lora = x_demo @ W_pretrained + (x_demo @ A) @ B * (8/4)
delta_lora = np.abs(out_lora - out_pretrained).mean()

print(f"Output change: full fine-tune = {delta_full:.4f}")
print(f"Output change: LoRA (r=4)     = {delta_lora:.4f}")
print("This chosen low-rank perturbation changes fewer directions than the chosen full update.")
print("At B=0 the adapter update is zero, but later behavior can still regress.")
print("Use a retention holdout; this arithmetic example does not prove protection.")


## 8 · Production Library Implementation


In [ ]:
# 8.1 HuggingFace tokenizer (BPE, production-grade).
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("gpt2")
    test = "Tokenization splits words into subword pieces efficiently."
    ids = tok.encode(test)
    decoded = [tok.decode([i]) for i in ids]
    print("GPT-2 BPE tokenizer:")
    print(f"  Input:  '{test}'")
    print(f"  Tokens: {decoded}")
    print(f"  IDs:    {ids}")
    print(f"  Vocab size: {tok.vocab_size}")
except Exception as e:
    print(f"[transformers not available: {type(e).__name__}]")
    print("In production: AutoTokenizer.from_pretrained('gpt2') for GPT-2 BPE,")
    print("or 'meta-llama/Llama-2-7b' for SentencePiece BPE with 32K vocab.")


In [ ]:
# 8.2 LoRA / PEFT with HuggingFace (guarded).
try:
    from peft import LoraConfig, get_peft_model, TaskType
    from transformers import AutoModelForCausalLM
    import torch

    base = AutoModelForCausalLM.from_pretrained("gpt2")
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        target_modules=["c_attn"],     # GPT-2 attention projection
        lora_dropout=0.05,
    )
    peft_model = get_peft_model(base, lora_cfg)
    trainable, total = peft_model.get_nb_trainable_parameters()
    print(f"PEFT LoRA model: {trainable:,} trainable / {total:,} total "
          f"({100*trainable/total:.3f}%)")
except Exception as e:
    print(f"[peft/transformers/torch not available: {type(e).__name__}]")
    print("Production pattern:")
    print("  from peft import LoraConfig, get_peft_model")
    print("  cfg = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj','v_proj'])")
    print("  model = get_peft_model(base_model, cfg)")
    print("  # Then train with standard Trainer or SFTTrainer from trl")


In [ ]:
# 8.3 DPO training pattern (trl library, guarded).
lines = [
    "DPO training with trl -- production pattern (guarded, requires GPU + model):",
    "  from trl import DPOTrainer, DPOConfig",
    "  from transformers import AutoModelForCausalLM, AutoTokenizer",
    "  ",
    "  model     = AutoModelForCausalLM.from_pretrained('llama3-8b')",
    "  ref_model = AutoModelForCausalLM.from_pretrained('llama3-8b')  # frozen ref",
    "  tokenizer = AutoTokenizer.from_pretrained('llama3-8b')",
    "  ",
    "  # dataset: columns 'prompt', 'chosen', 'rejected'",
    "  trainer = DPOTrainer(",
    "      model=model, ref_model=ref_model,",
    "      args=DPOConfig(beta=0.1, max_length=512),",
    "      train_dataset=pref_dataset, tokenizer=tokenizer,",
    "  )",
    "  trainer.train()",
    "",
    "Key: beta=0.1 controls KL-divergence strength from reference policy.",
]
print('\n'.join(lines))


## 9 · Realistic Business Case Study — Fine-Tuning a Domain LLM for Medical Q&A

**Scenario.** A healthtech company wants an LLM that answers clinician questions
about drug interactions and dosing. The base model (Llama-3-8B) is general; they
need domain accuracy without hallucination.

**Pipeline chosen:**
1. **Continual pre-training candidate** on licensed, de-identified clinical text.
   Estimate compute from a measured pilot rather than a generic hardware number.
2. **SFT** on 50K clinician-verified Q&A pairs in the ChatML format.
   Loss masked on prompt tokens.
3. **DPO alignment** on 10K preference pairs where clinicians rated two responses
   and marked one as more accurate/safe.

**Why compare DPO first?** PPO-based RLHF adds a reward model and online policy
optimization. DPO offers a simpler offline baseline. Neither is universally better;
compare task quality, safety slices, regressions, and operational cost.

**Why test LoRA?** It reduces optimizer and gradient state by updating adapters.
Hardware and duration depend on sequence length, batch, precision, and model. A
retention set is still required because frozen base weights do not prevent the
combined model from changing behavior.

**Cost of mistakes:** hallucinated drug dosage → patient harm → liability. This
mandates: (1) constitutional AI rules in the system prompt; (2) factual
grounding via RAG (Section 06); (3) LLM-as-judge eval on a medical golden set
(Lesson EVAL-05); (4) human expert review of random outputs weekly.

**KPIs:** factual accuracy on medical QA benchmark (MedQA), hallucination rate
(Lesson NLP-05), p95 latency, cost-per-query.


## 10 · Production Considerations

- **Measure data quality and quantity together.** Filtering often helps, but no
  fixed small dataset is universally sufficient. Compare controlled data slices.
- **Deduplication for pre-training.** Near-duplicate removal (MinHash, exact
  substring) is essential — duplicates inflate effective data size, encourage
  memorisation, and leak test data.
- **Gradient checkpointing + bf16.** Standard for fine-tuning on limited GPU:
  recompute activations during backward pass to save memory (at cost of ~30% more
  compute); use bfloat16 for numerical stability.
- **Learning-rate schedule.** Treat published values as starting hypotheses. Run a
  small sweep and diagnose divergence, underfitting, and retention on held-out data.
- **Evaluation during training.** Monitor: SFT loss on held-out set, perplexity,
  and task-specific metrics (ROUGE, accuracy). Early-stop on task metric.
- **Merging LoRA for inference.** With this notebook's row-vector convention, merge
  $W = W_0 + AB\cdot\alpha/r$
  back into the base model weights — zero inference overhead vs the base model.
- **System prompt as alignment.** For many production use cases, a well-crafted
  system prompt (Lesson NLP-04) combined with SFT is sufficient — RLHF/DPO adds
  complexity and cost that may not be justified.
- **Versioning.** Track: base model version, LoRA adapter checkpoint, tokeniser
  version, training data hash. A mismatch between any of these breaks behaviour
  silently.


## 11 · Tradeoff Analysis

**Fine-tuning strategy:**

| Strategy | Cost | Risk | Quality | When to use |
|---|---|---|---|---|
| Prompt engineering | Zero | None | Baseline | Quick experiments (NLP-04) |
| LoRA/PEFT | Lower trainable state | Behavior can still regress | Must measure | Resource-constrained adaptation |
| Full fine-tuning | Higher optimizer state | Behavior can regress | Must measure | Adapter capacity is measurably insufficient |
| RAG (no FT) | Near-zero | Staleness | Good for factual | Knowledge-heavy tasks (RAG-02) |

**Alignment strategy:**

| Strategy | RM needed | Stability | Cost | Quality |
|---|---|---|---|---|
| SFT only | No | High | Low | Good baseline |
| RLHF (PPO) | Yes | More moving parts | Higher | Supports online policy optimization |
| **DPO** | **No separate RM** | Simpler offline loop | Lower | Baseline for preference pairs |
| ORPO | No | Needs its own evaluation | Lower | Combines supervised and preference terms |

**LoRA rank selection:**

| Rank | Trainable % | Quality | Memory | When |
|---|---|---|---|---|
| r=1 | ~0.05% | Low | Minimal | Prompt style only |
| r=8 | Depends on target layers | Unknown before evaluation | Lower | First capacity candidate |
| r=16 | Depends on target layers | Unknown before evaluation | Higher | Compare when rank 8 underfits |
| r=64 | Depends on target layers | Unknown before evaluation | Higher | Only after measured capacity failure |


## 12 · Senior-Level Interview Preparation

**Common questions**
- *"Explain BPE tokenisation."* → Start with characters. Iteratively merge the
  most frequent adjacent pair into a new token. Stop at target vocab size. Encoding
  new text: apply merges greedily in training order. Why BPE: handles OOV via
  subwords, no character-level slowness, vocabulary scales with corpus richness.
- *"SFT vs RLHF vs DPO?"* → SFT: teach format and domain via supervised examples.
  RLHF: learn human preferences via reward model + PPO. DPO: same objective as RLHF
  but reformulated to eliminate the RM — train directly on preference pairs.

**Deep-dive questions**
- *"Why does DPO work without a reward model?"* → The RLHF optimal policy
  $\pi^*(y|x) \propto \pi_{\text{ref}} \exp(r/\beta)$ can be inverted to express
  $r$ as a function of $\pi^*$ and $\pi_{\text{ref}}$. Plugging into the Bradley-
  Terry preference model gives a loss purely in terms of policy log-ratios — no RM.
- *"Does LoRA prevent catastrophic forgetting?"* → No guarantee. $W_0$ is frozen
  and the adapter begins with zero effective update, but the combined model can
  still change behavior. Measure task quality and retention on separate holdouts.
- *"What is loss masking in SFT?"* → We compute the cross-entropy only on response
  tokens, not prompt tokens. If we trained on prompt tokens too, the model would
  optimize prompt reconstruction as an additional objective. The prompt remains
  conditioning context, and response losses can still backpropagate through its
  hidden representations.

**Whiteboard questions**
- "Write the DPO loss and explain each term." (§4.5)
- "Draw the BPE merge algorithm step-by-step for the word 'cat'." (§4.1, §5a)

**Strong vs weak answers**
- *"When should we use RLHF vs DPO?"*
  - **Weak:** "RLHF is always better."
  - **Strong:** "DPO is a simpler offline baseline because it avoids a separate
    reward model and PPO loop. PPO-based RLHF supports online policy optimization
    but adds moving parts. Compare both only when the task, preference data, safety
    slices, and operational budget justify the experiment."

**Common mistakes:** thinking BPE is character-level (it's subword); confusing SFT
loss (response-only) with pre-training loss (all tokens); not knowing LoRA merging
at inference; thinking RLHF requires human labels at inference time (no — RM is
trained offline).


## 13 · Teach-Back — Answer Without Notes

1. **BPE in one minute.** Walk through the merge algorithm from character tokens to
   subwords. What does the vocabulary contain at the end?
2. **CLM loss.** Write the formula. What does a loss of 2.0 mean in terms of bits
   per token?
3. **SFT masking.** Why do we mask the prompt tokens from the loss? What goes wrong
   if we don't?
4. **RLHF stages.** Name the four steps. What is the reward model trained on?
5. **DPO.** What is $\pi_{\text{ref}}$ and why is it needed? What does $\beta$
   control?
6. **LoRA.** Write $h = xW_0 + x(AB)\alpha/r$. Why is $B$ initialised to zero?
7. **Forgetting.** Why can a LoRA-adapted model regress even when $W_0$ is frozen?
8. **When to use what.** Your company has 10K instruction-response pairs and 2 A100
   GPUs. What pipeline do you run?


## 14 · Exercises

**Beginner (conceptual)**
1. Trace BPE on the word "running": start from ['r','u','n','n','i','n','g','</w>']
   and show 3 merge steps, choosing the pair "n n" as the first merge.
2. What is the expected cross-entropy loss for a 50K-vocabulary model at random
   initialisation? Compute it.

**Beginner → Intermediate (coding)**
3. Extend the BPE implementation to encode unseen words by applying the learned
   merges in order. Encode "fine-tuning" and compare its tokens to "tuning" and
   "fine".
4. Add loss masking to the causal LM loss: accept a `mask` array where 0 = ignored,
   1 = included. Verify that masking all prompt tokens produces the SFT loss.

**Intermediate (analysis)**
5. Implement the RLHF reward model training loss (Bradley-Terry, §4.4) from scratch.
   Generate synthetic (winner, loser) log-probability pairs and train a simple linear
   reward head to predict preference. Measure accuracy on a held-out set.
6. Show that LoRA with rank $r=d$ is equivalent to full fine-tuning (theoretically
   and by checking that the output matches a direct $\Delta W$ update when $A$ and
   $B$ are set appropriately).

**Senior (interview + production design)**
7. *Design:* the full training pipeline for the medical Q&A LLM in §9. Include:
   data curation (dedup, quality filter), continual pre-training compute budget
   (Chinchilla), SFT data size and format, DPO preference collection strategy,
   evaluation (EVAL-02, EVAL-04, and EVAL-05), and A/B rollout plan.
8. *Scaling:* you have 64 A100 GPUs and a target validation loss of 2.5 nat.
   Using Chinchilla laws, estimate the model size and token count for optimal
   compute use. Then estimate how long training takes assuming 312 TFLOP/s per A100
   at 40% MFU.
9. *Debugging:* after DPO fine-tuning, your model's outputs become shorter and
   more generic. Identify two likely causes from §7 and propose concrete mitigations.


---
### Summary
The LLM training pipeline has four stages: **pre-training** (learn language from
internet-scale text via next-token prediction, budget by Chinchilla), **continual
pre-training** (domain adaptation), **SFT** (teach instruction-following via
response-only cross-entropy), and **alignment** (RLHF via PPO+RM, or DPO via
direct preference pairs with a simpler offline loop). **LoRA** can reduce trainable
state by updating low-rank adapter matrices while keeping base weights frozen; its
quality and retention still require direct evaluation.

**Related lesson:** `NLP-04 · Prompt Engineering` — the other side of the LLM interface: how
to construct system prompts, few-shot examples, chain-of-thought, and structured
output to maximise model capability without fine-tuning.


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **NLP-03 · LLM Training Pipeline** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember NLP-03 · LLM Training Pipeline: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · NLP-03

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
